# Step 4: tables and statistics behind the figures

Per-nucleus tables, edge shares, label agreement, and the permutation control of the pseudotime transfer.

In [ ]:
OUT = ""  # run folder
SAMAP_FINAL = ""  # default: OUT/3_transfer/Samap_Final.h5ad (step 3)
HUMAN_H5AD = "/n/netscratch/eddy_lab/Everyone/nhilgert/final_objects/Cevrim_Human_Only_Fib_Dec_Cells.h5ad"
MOUSE_H5AD = "/n/netscratch/eddy_lab/Everyone/nhilgert/final_objects/Cevrim_XMens_Only_Fib_Dec_Cells.h5ad"
X_COLUMN = "x_um_qc"  # the human object's section coordinates
Y_COLUMN = "y_um_qc"
SEED = 20261022
N_PERM = 1000

In [ ]:
import datetime
import hashlib
import json
import os
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import scipy.sparse as sp
from anndata.io import read_elem
from scipy.stats import rankdata
from sklearn.neighbors import NearestNeighbors

In [ ]:
assert OUT, "set OUT, the run folder"
samap_final = Path(SAMAP_FINAL or Path(OUT) / "3_transfer" / "Samap_Final.h5ad")
out_dir = Path(OUT) / "4_statistics"
partial_dir = Path(OUT) / "4_statistics.partial"
assert not out_dir.exists(), f"{out_dir} exists; step 4 is done"
for folder in ("tables", "graph_block", "edge_share", "ptnull"):
    (partial_dir / folder).mkdir(parents=True, exist_ok=True)
statistics = {"written": datetime.datetime.now().isoformat(timespec="seconds"), "samap_final": str(samap_final)}


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 24), b""):
            digest.update(block)
    return digest.hexdigest()


# The objects label the compartments eFib and dec; later steps use the long names.
LABEL_MAP = {"eFib": "endometrial_fibroblast", "dec": "decidual_cell"}


def final_obs(h5ad):
    """obs of a final object, indexed by nucleus."""
    with h5py.File(h5ad, "r") as f:
        obs = read_elem(f["obs"])
    obs.index.name = "cell"
    obs["cell_type"] = obs["cell_type"].astype(str).map(lambda v: LABEL_MAP.get(v, v)).astype("category")
    return obs

## 1. Per-nucleus tables

In [ ]:
with h5py.File(samap_final, "r") as f:
    joint_obs = read_elem(f["obs"])
    umap = f["obsm/X_umap"][:]
    graphs = {key: sp.csr_matrix(read_elem(f["obsp"][key])) for key in f["obsp"].keys()}
    mapping_k = f["uns/mapping_K"][()].item() if "uns/mapping_K" in f else None

joint = pd.DataFrame({
    "species": joint_obs["species"],
    "sample": joint_obs["sample"],
    "timepoint": joint_obs["timepoint"],
    "transferred_mm_dpt_pseudotime_ranknorm": joint_obs["transferred_mouse_pseudotime"],
}, index=pd.Index(joint_obs.index, name="cell"))
joint["X_umap__0"] = umap[:, 0]
joint["X_umap__1"] = umap[:, 1]

mouse_obs = final_obs(MOUSE_H5AD)
human_obs = final_obs(HUMAN_H5AD)
# The tables use the column names the figure notebooks read.
human_obs = human_obs.rename(columns={X_COLUMN: "x_um_qc", Y_COLUMN: "y_um_qc"})
for code, obs in (("mm", mouse_obs), ("hs", human_obs)):
    in_joint = joint.index[joint["species"].astype(str) == code]
    assert len(in_joint) == len(obs) and in_joint.isin(obs.index).all(), f"{code}: the final object and Samap_Final differ"

mouse_cells = mouse_obs[["cell_type", "cell_subtype", "pseudotime"]].rename(columns={"pseudotime": "pt_harmonized_rank"})
human_cells = human_obs[["sample", "timepoint", "cell_type", "x_um_qc", "y_um_qc"]].copy()
human_cells["pt_transferred"] = joint.loc[human_cells.index, "transferred_mm_dpt_pseudotime_ranknorm"].to_numpy()
human_cells["pt_human"] = human_obs["pseudotime"].to_numpy()

joint.to_parquet(partial_dir / "tables" / "joint.parquet")
mouse_cells.to_csv(partial_dir / "tables" / "mouse_cells.csv")
human_cells.to_csv(partial_dir / "tables" / "human_cells.csv")
human_obs.to_parquet(partial_dir / "tables" / "human_plot_table.parquet")

statistics["nuclei"] = {
    "joint": int(len(joint)), "mouse": int(len(mouse_cells)), "human": int(len(human_cells)),
    "mouse_with_pseudotime": int(mouse_cells["pt_harmonized_rank"].notna().sum()),
    "human_with_transferred_pseudotime": int(human_cells["pt_transferred"].notna().sum()),
    "human_positioned": int(human_cells["x_um_qc"].notna().sum()),
}
print(json.dumps(statistics["nuclei"]))

## 2. The human x mouse block of the joint graph

In [ ]:
names = joint.index.astype(str).to_numpy()
species = joint["species"].astype(str).to_numpy()
is_human, is_mouse = species == "hs", species == "mm"
C = graphs["connectivities"][is_human][:, is_mouse].tocsr()
sp.save_npz(partial_dir / "graph_block" / "C_hs_mm.npz", C, compressed=True)
(partial_dir / "graph_block" / "hs_names.txt").write_text("\n".join(names[is_human]) + "\n")
(partial_dir / "graph_block" / "mm_names.txt").write_text("\n".join(names[is_mouse]) + "\n")

row_edges = np.diff(C.indptr)
graph_meta = {
    "joint": str(samap_final),
    "joint_bytes": samap_final.stat().st_size,
    "joint_sha256": sha256_of(samap_final),
    "mapping_K": mapping_k,
    "n_human": int(is_human.sum()),
    "n_mouse": int(is_mouse.sum()),
    "nnz": int(C.nnz),
    "row_nnz_min": int(row_edges.min()),
    "row_nnz_median": float(np.median(row_edges)),
    "row_nnz_max": int(row_edges.max()),
    "rows_without_mouse_edge": int((row_edges == 0).sum()),
}
(partial_dir / "graph_block" / "meta.json").write_text(json.dumps(graph_meta, indent=1))
statistics["graph_block"] = graph_meta
print(json.dumps(graph_meta))

## 3. Edge shares

The share of each compartment's cross-species edge weight that reaches each compartment of the other species.

In [ ]:
COMPARTMENTS = ["endometrial_fibroblast", "decidual_cell"]


def edge_shares(graph, species, compartment):
    """One row per direction and source compartment: summed and per-nucleus shares of cross-species edge weight."""
    rows = []
    for source, target, direction in (("hs", "mm", "human_to_mouse"), ("mm", "hs", "mouse_to_human")):
        source_idx = np.flatnonzero(species == source)
        target_idx = np.flatnonzero(species == target)
        cross = graph[source_idx][:, target_idx]
        target_compartment = compartment[target_idx]
        to_compartment = np.column_stack([
            np.asarray(cross[:, target_compartment == c].sum(axis=1)).ravel() for c in COMPARTMENTS
        ])
        for c in COMPARTMENTS:
            in_compartment = compartment[source_idx] == c
            summed = to_compartment[in_compartment].sum(axis=0)
            totals = to_compartment[in_compartment].sum(axis=1)
            # Nuclei without a cross-species edge have no share; they count only in n_source_cells.
            with_edges = totals > 0
            per_cell = (to_compartment[in_compartment][with_edges] / totals[with_edges, None]).mean(axis=0)
            rows.append({
                "direction": direction,
                "source_compartment": c,
                "n_source_cells": int(in_compartment.sum()),
                "n_with_cross_edges": int(with_edges.sum()),
                **{f"weight_share_{t}": float(s) for t, s in zip(COMPARTMENTS, summed / summed.sum())},
                **{f"percell_share_{t}": float(s) for t, s in zip(COMPARTMENTS, per_cell)},
            })
    return rows


tables_dir = partial_dir / "tables"
release_labels = pd.concat([
    pd.read_csv(tables_dir / "human_cells.csv", index_col="cell", usecols=["cell", "cell_type"])["cell_type"],
    pd.read_csv(tables_dir / "mouse_cells.csv", index_col="cell", usecols=["cell", "cell_type"])["cell_type"],
]).reindex(joint.index)
assert release_labels.notna().all(), "joint nuclei without a label"

rows = []
for graph_name, graph in graphs.items():
    for row in edge_shares(graph, species, release_labels.astype(str).to_numpy()):
        rows.append({"graph": graph_name, "labels": "release", **row})
edge_table = pd.DataFrame(rows)
edge_table.to_csv(partial_dir / "edge_share" / "samap_compartment_edge_share_all.csv", index=False)

figure_table = edge_table[edge_table["graph"] == "connectivities"].set_index(["direction", "source_compartment"])
figure_table = figure_table[[f"weight_share_{c}" for c in COMPARTMENTS]]
figure_table.columns = COMPARTMENTS
figure_table.to_csv(partial_dir / "edge_share" / "samap_compartment_edge_share.csv")
statistics["edge_share"] = {f"{d}/{s}": row.round(4).to_dict() for (d, s), row in figure_table.iterrows()}
print(figure_table.round(3).to_string())

## 3b. Cross-species edge weight per nucleus, and SAMap's alignment scores

In [ ]:
graph = graphs["connectivities"].copy()
graph.eliminate_zeros()
labels = release_labels.astype(str).to_numpy()
assert mapping_k, f"{samap_final} has no uns['mapping_K']"

weight_rows = []
into = {}  # (source species, source compartment, target compartment) -> mean edge weight per source nucleus
for source, target, direction in (("hs", "mm", "human_to_mouse"), ("mm", "hs", "mouse_to_human")):
    source_idx, target_idx = np.flatnonzero(species == source), np.flatnonzero(species == target)
    cross = graph[source_idx][:, target_idx].tocsr()
    to_compartment = np.column_stack([
        np.asarray(cross[:, labels[target_idx] == c].sum(axis=1)).ravel() for c in COMPARTMENTS
    ])
    total = to_compartment.sum(axis=1)
    n_edges = np.diff(cross.indptr)
    for c in COMPARTMENTS:
        members = labels[source_idx] == c
        weight_rows.append({
            "direction": direction,
            "source_compartment": c,
            "n_source_cells": int(members.sum()),
            "n_without_cross_edge": int((total[members] == 0).sum()),
            "mean_total_weight": float(total[members].mean()),
            "max_total_weight": float(total[members].max()),
            "median_cross_edges": float(np.median(n_edges[members])),
            "max_cross_edges": int(n_edges[members].max()),
        })
        for k, t in enumerate(COMPARTMENTS):
            into[(source, c, t)] = float(to_compartment[members, k].mean())
weight_table = pd.DataFrame(weight_rows)
weight_table.to_csv(partial_dir / "edge_share" / "samap_compartment_edge_weight.csv", index=False)

alignment_rows = []
for human_compartment in COMPARTMENTS:
    for mouse_compartment in COMPARTMENTS:
        human_to_mouse = into[("hs", human_compartment, mouse_compartment)] / mapping_k
        mouse_to_human = into[("mm", mouse_compartment, human_compartment)] / mapping_k
        alignment_rows.append({"human_compartment": human_compartment, "mouse_compartment": mouse_compartment,
                               "human_to_mouse": human_to_mouse, "mouse_to_human": mouse_to_human,
                               "alignment_score": max(human_to_mouse, mouse_to_human)})
alignment_table = pd.DataFrame(alignment_rows)
alignment_table.to_csv(partial_dir / "alignment_scores.csv", index=False)

statistics["edge_weight"] = {
    f"{row['direction']}/{row['source_compartment']}": {
        key: round(value, 4) if isinstance(value, float) else value
        for key, value in row.items() if key not in ("direction", "source_compartment")}
    for row in weight_rows}
statistics["alignment_scores"] = {f"{row['human_compartment']}/{row['mouse_compartment']}": round(row["alignment_score"], 4)
                                  for row in alignment_rows}
print(weight_table.round(3).to_string(index=False))
print(alignment_table.round(4).to_string(index=False))

## 4. Agreement of the transferred cell type with the human labels

In [ ]:
human_rows = joint_obs["species"].astype(str) == "hs"
native = joint_obs.loc[human_rows, "cell_type"].astype(str)
transferred = joint_obs.loc[human_rows, "transferred_mouse_fibdec"]
labelled = transferred.notna()
agree = labelled & (transferred.astype(str) == native)

rows = []
for group, members in [("all", native.index), *[(c, native.index[native == c]) for c in sorted(native.unique())]]:
    rows.append({
        "native_cell_type": group,
        "nuclei": int(len(members)),
        "labelled": int(labelled[members].sum()),
        "agree": int(agree[members].sum()),
        "agree_share_of_labelled": round(float(agree[members].sum() / labelled[members].sum()), 4),
        "agree_share_of_all": round(float(agree[members].mean()), 4),
    })
agreement = pd.DataFrame(rows)
agreement.to_csv(partial_dir / "label_agreement.csv", index=False)
statistics["label_agreement"] = agreement.set_index("native_cell_type").to_dict(orient="index")
print(agreement.to_string(index=False))

## 5. Permutation control of the pseudotime transfer

Mouse pseudotime shuffled across all mouse nuclei, or within each mouse compartment, 1,000 times each.

In [ ]:
MOUSE_PT = "pt_harmonized_rank"  # transferred
HUMAN_PT = "pt_human"  # compared with
SPATIAL_K = 10  # spatial neighbors per nucleus for Moran's I
CHUNK = 250  # replicates per matrix product
REPORTED = ["rho_all", "rho_decidual", "rho_fibroblast", "moran_all", "moran_within_decidual",
            "moran_within_fibroblast"]


def transfer(C, values, has_value):
    """Kernel-weighted average of mouse values (n_mouse x R; 0 where a nucleus has no value) onto the human rows."""
    numerator = C @ values
    denominator = C @ has_value.astype(float)
    out = np.full(numerator.shape, np.nan)
    ok = denominator > 1e-10
    out[ok] = numerator[ok] / denominator[ok][:, None]
    return out


def rank_unit(values, has_value):
    """Average rank / N over the nuclei with a value; 0 elsewhere."""
    out = np.zeros(len(values))
    out[has_value] = rankdata(values[has_value]) / has_value.sum()
    return out


def spearman_columns(transferred, human_pt):
    """Spearman rho between human_pt and each column of transferred."""
    rank_human = rankdata(human_pt)
    rank_human = (rank_human - rank_human.mean()) / rank_human.std()
    ranks = rankdata(transferred, axis=0)
    ranks = (ranks - ranks.mean(0)) / ranks.std(0)
    return (ranks * rank_human[:, None]).mean(0)


def knn_weights(coords):
    """Row-standardized weights on each nucleus's SPATIAL_K nearest neighbors (positions in microns)."""
    nn = NearestNeighbors(n_neighbors=SPATIAL_K + 1).fit(coords)
    _, idx = nn.kneighbors(coords)
    rows = np.repeat(np.arange(len(coords)), SPATIAL_K)
    cols = idx[:, 1:].ravel()  # column 0 is the nucleus itself
    data = np.full(rows.size, 1.0 / SPATIAL_K)
    return sp.csr_matrix((data, (rows, cols)), shape=(len(coords), len(coords)))


def morans_i(W, values):
    """Moran's I of each column; with row-standardized W, z'Wz / z'z."""
    z = values - values.mean(0)
    return (z * (W @ z)).sum(0) / (z * z).sum(0)

In [ ]:
ptnull_dir = partial_dir / "ptnull"
rng = np.random.default_rng(SEED)

# Inputs in the graph's row and column order.
graph_dir = partial_dir / "graph_block"
C = sp.load_npz(graph_dir / "C_hs_mm.npz").tocsr()
hs_names = (graph_dir / "hs_names.txt").read_text().split()
mm_names = (graph_dir / "mm_names.txt").read_text().split()
mouse = pd.read_csv(tables_dir / "mouse_cells.csv", index_col="cell").loc[mm_names]
human = pd.read_csv(tables_dir / "human_cells.csv", index_col="cell").loc[hs_names]

# The observed transfer. Nuclei without a value stay out of the numerator and the denominator.
has_value = mouse[MOUSE_PT].notna().to_numpy()
pt_rank = rank_unit(mouse[MOUSE_PT].to_numpy(), has_value)
observed = transfer(C, pt_rank[:, None], has_value)[:, 0]

# The recomputed transfer must equal the stored one before any shuffle is drawn.
stored = human["pt_transferred"].to_numpy()
same_nan = np.array_equal(np.isnan(observed), np.isnan(stored))
max_diff = float(np.nanmax(np.abs(observed - stored)))
print(f"recomputed transfer: NaN pattern identical {same_nan}, max |diff| {max_diff:.2e}")
assert same_nan and max_diff <= 1e-9, "the recomputed transfer differs from the stored one"

human_pt = human[HUMAN_PT].to_numpy()
scored = np.isfinite(observed) & np.isfinite(human_pt)
subsets = {
    "all": scored,
    "decidual": scored & (human["cell_type"] == "decidual_cell").to_numpy(),
    "fibroblast": scored & (human["cell_type"] == "endometrial_fibroblast").to_numpy(),
}

# Each timepoint is one tissue section. Moran's I is computed per section and averaged.
positioned = scored & human["x_um_qc"].notna().to_numpy()
sections = {t: np.flatnonzero(positioned & (human["timepoint"] == t).to_numpy()) for t in sorted(human["timepoint"].unique())}
weights = {t: knn_weights(human.iloc[ix][["x_um_qc", "y_um_qc"]].to_numpy()) for t, ix in sections.items()}

# Within a compartment, neighbors come only from that compartment's nuclei.
compartment_cells = {(k, t): ix[subsets[k][ix]] for k in ("decidual", "fibroblast") for t, ix in sections.items()}
compartment_weights = {key: knn_weights(human.iloc[ix][["x_um_qc", "y_um_qc"]].to_numpy())
                       for key, ix in compartment_cells.items()}
print({k: int(v.sum()) for k, v in subsets.items()}, {t: len(ix) for t, ix in sections.items()})


def replicate_statistics(transferred):
    stats = {f"rho_{k}": spearman_columns(transferred[m], human_pt[m]) for k, m in subsets.items()}
    for t, ix in sections.items():
        stats[f"moran_{t}"] = morans_i(weights[t], transferred[ix])
    stats["moran_all"] = np.mean([stats[f"moran_{t}"] for t in sections], axis=0)
    for k in ("decidual", "fibroblast"):
        per_section = [morans_i(compartment_weights[(k, t)], transferred[compartment_cells[(k, t)]]) for t in sections]
        stats[f"moran_within_{k}"] = np.mean(per_section, axis=0)
    stats["sd_raw"] = np.nanstd(transferred[scored], axis=0)
    return pd.DataFrame(stats)

In [ ]:
# uniform: shuffled across all mouse nuclei with a value.
# within_compartment: shuffled among the nuclei of one mouse cell type.
groups = {
    "uniform": [np.flatnonzero(has_value)],
    "within_compartment": [np.flatnonzero(has_value & (mouse["cell_type"] == ct).to_numpy())
                           for ct in sorted(mouse["cell_type"].unique())],
}
tables = [replicate_statistics(observed[:, None]).assign(design="observed", replicate=0)]
examples = {}
for design, index_groups in groups.items():
    for start in range(0, N_PERM, CHUNK):
        n = min(CHUNK, N_PERM - start)
        permuted = np.zeros((len(mouse), n))
        for r in range(n):
            for ix in index_groups:
                permuted[ix, r] = pt_rank[rng.permutation(ix)]
        transferred = transfer(C, permuted, has_value)
        if start == 0:
            examples[design] = transferred[:, 0]
        tables.append(replicate_statistics(transferred).assign(design=design, replicate=np.arange(start, start + n) + 1))
    print(f"{design}: {N_PERM} replicates done")

human_stats = replicate_statistics(human_pt[:, None])  # Moran's I of the human pseudotime itself
replicates = pd.concat(tables, ignore_index=True)
replicates.to_csv(ptnull_dir / "null_replicates.csv", index=False)
pd.DataFrame({"cell": hs_names, "observed": observed, **{f"{d}_example": v for d, v in examples.items()}}).to_csv(
    ptnull_dir / "example_transfers.csv", index=False)

metric_columns = [c for c in replicates.columns if c.startswith(("rho_", "moran_", "sd_"))]
null_summary = {
    "n_perm": N_PERM,
    "seed": SEED,
    "spatial_k": SPATIAL_K,
    "mouse_pt_col": MOUSE_PT,
    "human_pt_col": HUMAN_PT,
    "reproduction_max_abs_diff": max_diff,
    "n_scored": {k: int(v.sum()) for k, v in subsets.items()},
    "human_pt_moran": human_stats.filter(like="moran_").iloc[0].round(4).to_dict(),
    "by_design": {
        d: {c: {"mean": round(float(g[c].mean()), 4), "sd": round(float(g[c].std()), 4),
                "max": round(float(g[c].max()), 4)} for c in metric_columns}
        for d, g in replicates.groupby("design")
    },
}
(ptnull_dir / "null_summary.json").write_text(json.dumps(null_summary, indent=1))

observed_row = replicates[replicates["design"] == "observed"].iloc[0]
rows = []
for stat in REPORTED:
    row = {"statistic": stat, "observed": observed_row[stat]}
    for design in groups:
        values = replicates.loc[replicates["design"] == design, stat]
        row |= {f"{design}_mean": values.mean(), f"{design}_max": values.max(),
                f"{design}_n_ge_obs": int((values >= observed_row[stat]).sum())}
    rows.append(row)
ptnull_table = pd.DataFrame(rows).round(3)
ptnull_table.to_csv(ptnull_dir / "ptnull_statistics_table.csv", index=False)
statistics["ptnull"] = {"table": ptnull_table.set_index("statistic").to_dict(orient="index"),
                        "human_pt_moran": null_summary["human_pt_moran"],
                        "observed_moran_by_section": {t: round(float(observed_row[f"moran_{t}"]), 4) for t in sections}}
print(ptnull_table.to_string(index=False))
print("human pseudotime, Moran's I:", null_summary["human_pt_moran"])

In [ ]:
def plain(value):
    """numpy numbers as plain Python ones (integers stay integers), for json."""
    return value.item() if hasattr(value, "item") else str(value)


(partial_dir / "statistics.json").write_text(json.dumps(statistics, indent=1, default=plain) + "\n")
os.replace(partial_dir, out_dir)
print(f"wrote {out_dir}: {sorted(p.name for p in out_dir.iterdir())}")